<a href="https://colab.research.google.com/github/xwang335/Campbell-A/blob/main/enet_huber(backtest_version).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ENet+H replication notebook (paper-aligned, lambda floor retained)

This version is cleaned up so the prediction parquet is directly compatible with the backtest notebook.

Main choices:
- Expanding train + rolling 12-year validation
- Refit once per year
- `rho = 0.5` fixed
- `xi` fixed each year at the 99.9th percentile of validation residuals from a ridge fit
- `lambda` selected by validation Huber loss
- **Necessary lambda floor retained on purpose**

In [1]:
import os
import gc
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

try:
    from google.colab import drive
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print("GPU:", torch.cuda.get_device_name(0))
else:
    DEVICE = torch.device("cpu")
    print("Using CPU")

print("Device:", DEVICE)
print("Torch:", torch.__version__)

Mounted at /content/drive
GPU: NVIDIA L4
Device: cuda
Torch: 2.10.0+cu128


In [2]:
# -------- paths --------
PARQUET_PATH = "/content/drive/MyDrive/industry_project/preprocess_data.parquet"
OUTPUT_DIR   = "/content/drive/MyDrive/backtest"
OUTPUT_PATH  = f"{OUTPUT_DIR}/enet_huber_results.parquet"
YEAR_INFO_PATH = f"{OUTPUT_DIR}/enet_huber_year_info.csv"

os.makedirs(OUTPUT_DIR, exist_ok=True)

df = pd.read_parquet(PARQUET_PATH)
df["DATE"] = pd.to_datetime(df["DATE"])

print("shape:", df.shape)
print("date range:", df["DATE"].min().date(), "to", df["DATE"].max().date())
print("permno count:", df["permno"].nunique())
df.head(3)

shape: (3712808, 109)
date range: 1957-04-30 to 2016-12-30
permno count: 29825


,permno,DATE,mvel1,beta,betasq,chmom,dolvol,idiovol,indmom,mom1m,...,exret,exret_lead1,tbl,d/p,e/p,b/m,tms,dfy,ntis,svar
0,10000,1986-02-28,-0.381006,0.0,0.0,0.0,0.000000,0.0,0.310144,0.000485,...,-0.262443,0.359385,0.0707,0.037492,0.068845,0.583517,0.0251,0.0139,-0.019172,0.001920
1,10000,1986-03-31,-0.505829,0.0,0.0,0.0,0.000000,0.0,0.358323,-0.952720,...,0.359385,-0.103792,0.0706,0.035167,0.064120,0.536377,0.0135,0.0144,-0.017914,0.001089
2,10000,1986-04-30,-0.410132,0.0,0.0,0.0,-0.539206,0.0,0.281704,0.932882,...,-0.103792,-0.227556,0.0656,0.033571,0.060779,0.519628,0.0110,0.0150,-0.016420,0.001374


In [3]:
# -------- column sets --------
NON_CHAR_COLS = {
    "permno", "DATE", "ret", "rf", "exret", "exret_lead1", "sic2",
    "tbl", "b/m", "d/p", "e/p", "ntis", "tms", "dfy", "svar"
}
MACRO_COLS = ["tbl", "b/m", "d/p", "e/p", "ntis", "tms", "dfy", "svar"]
CHAR_COLS_94 = sorted([c for c in df.columns if c not in NON_CHAR_COLS])

sic2_dummies = pd.get_dummies(df["sic2"].astype(str), prefix="sic2", drop_first=False)
SIC2_DUMMY_COLS = sorted(sic2_dummies.columns.tolist())
df = pd.concat([df, sic2_dummies], axis=1)
del sic2_dummies
gc.collect()

print("94 stock characteristics:", len(CHAR_COLS_94))
print("8 macro variables:", MACRO_COLS)
print("SIC2 dummy cols:", len(SIC2_DUMMY_COLS))
print("total features:", len(CHAR_COLS_94) * 9 + len(SIC2_DUMMY_COLS))

94 stock characteristics: 94
8 macro variables: ['tbl', 'b/m', 'd/p', 'e/p', 'ntis', 'tms', 'dfy', 'svar']
SIC2 dummy cols: 74
total features: 920


In [4]:
# -------- model config --------
TARGET = "exret_lead1"
VALIDATION_END = pd.Timestamp("1986-12-31")
TEST_END = pd.Timestamp("2016-12-31")
VALIDATION_YEARS = 12

# paper grid is [1e-4, 1e-1], but we intentionally keep the lambda floor
_full_grid = np.logspace(-4, -1, 20)
LAM_GRID = _full_grid[_full_grid >= 1.26e-3]
LAM_GRID_SORTED = np.sort(LAM_GRID)[::-1]

RHO = 0.5
XI_QUANTILE = 0.999

REQUIRED = CHAR_COLS_94 + MACRO_COLS + [TARGET, "mvel1"]
df_clean = df.dropna(subset=REQUIRED).copy()
df_clean = df_clean.sort_values(["DATE", "permno"]).reset_index(drop=True)

test_years = sorted(
    df_clean.loc[
        (df_clean["DATE"] > VALIDATION_END) & (df_clean["DATE"] <= TEST_END),
        "DATE"
    ].dt.year.unique()
)

print("clean rows:", f"{len(df_clean):,}")
print("test years:", test_years[0], "to", test_years[-1], f"({len(test_years)})")
print("lambda grid:", LAM_GRID)

clean rows: 3,712,808
test years: 1987 to 2016 (30)
lambda grid: [0.00127427 0.00183298 0.00263665 0.00379269 0.00545559 0.0078476
 0.01128838 0.01623777 0.02335721 0.03359818 0.0483293  0.06951928
 0.1       ]


In [5]:
def build_features_920(data, char_cols, macro_cols, sic2_dummies_cols):
    chars = data[char_cols].to_numpy(dtype=np.float32)
    macro_with_const = np.column_stack([
        np.ones(len(data), dtype=np.float32),
        data[macro_cols].to_numpy(dtype=np.float32)
    ])
    interactions = (chars[:, :, None] * macro_with_const[:, None, :]).reshape(len(data), -1)
    sic2_vals = data[sic2_dummies_cols].to_numpy(dtype=np.float32)
    return np.hstack([interactions, sic2_vals])

def standardize(X, mean=None, std=None):
    if mean is None:
        mean = X.mean(axis=0)
    if std is None:
        std = X.std(axis=0)
    std = np.where(std < 1e-8, 1.0, std)
    return (X - mean) / std, mean, std

def huber_loss_t(residuals, xi):
    abs_r = residuals.abs()
    return torch.where(abs_r <= xi, residuals ** 2, 2.0 * xi * abs_r - xi ** 2)

def huber_grad_t(X, y, theta, xi):
    n = y.shape[0]
    residuals = y - X @ theta
    dH = torch.where(residuals.abs() <= xi, 2.0 * residuals, 2.0 * xi * residuals.sign())
    return -(X.T @ dH) / n

def prox_enet_t(theta, gamma, lam, rho):
    tau = gamma * lam * (1.0 - rho)
    shrunk = theta.sign() * torch.clamp(theta.abs() - tau, min=0.0)
    return shrunk / (1.0 + gamma * lam * rho)

def enet_objective_t(X, y, theta, lam, rho, xi):
    n = y.shape[0]
    loss = huber_loss_t(y - X @ theta, xi).sum() / n
    pen = lam * ((1.0 - rho) * theta.abs().sum() + 0.5 * rho * (theta ** 2).sum())
    return (loss + pen).item()

def apg_huber_enet_gpu(X, y, lam, rho, xi, gamma, max_iter=2000, tol=1e-5, theta_init=None, return_info=False):
    p = X.shape[1]
    theta = theta_init.clone() if theta_init is not None else torch.zeros(p, dtype=torch.float32, device=X.device)
    theta_old = theta.clone()

    converged = False
    final_diff = float("inf")
    iters_used = 0
    n_restarts = 0
    momentum_age = 0

    prev_obj = enet_objective_t(X, y, theta, lam, rho, xi)

    for m in range(max_iter):
        grad = huber_grad_t(X, y, theta, xi)
        theta_tilde = theta - gamma * grad
        theta_bar = prox_enet_t(theta_tilde, gamma, lam, rho)

        beta = momentum_age / (momentum_age + 3.0)
        theta_new = theta_bar + beta * (theta_bar - theta_old)

        curr_obj = enet_objective_t(X, y, theta_new, lam, rho, xi)
        if curr_obj > prev_obj:
            theta_new = theta_bar
            curr_obj = enet_objective_t(X, y, theta_new, lam, rho, xi)
            momentum_age = 0
            n_restarts += 1
        else:
            momentum_age += 1
        prev_obj = curr_obj

        diff = (theta_new - theta).norm().item()
        final_diff = diff
        iters_used = m + 1

        if diff < tol * (1.0 + theta.norm().item()):
            converged = True
            theta = theta_new
            break

        theta_old = theta.clone()
        theta = theta_new

    if return_info:
        return theta, {
            "iters": iters_used,
            "converged": converged,
            "final_diff": final_diff,
            "n_restarts": n_restarts,
        }
    return theta

def oos_r2(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    denom = np.sum(y_true ** 2)
    if denom == 0:
        return np.nan
    return 1.0 - np.sum((y_true - y_pred) ** 2) / denom

print("helpers ready")

helpers ready


## Main estimation loop

Paper-aligned split:
- train: 1957 to `year-13`
- validation: `year-12` to `year-1`
- test: `year`

And then:
- standardize with train moments only
- center `y` with train mean only
- fix `xi` from ridge residuals on validation
- tune `lambda` on validation
- refit on train only

In [6]:
all_preds = []
year_info = []
theta_cross_year = None

for year in test_years:
    t0 = time.time()

    train_end_year = year - VALIDATION_YEARS - 1
    val_start_year = year - VALIDATION_YEARS
    val_end_year = year - 1

    df_train = df_clean[df_clean["DATE"].dt.year <= train_end_year].copy()
    df_val   = df_clean[(df_clean["DATE"].dt.year >= val_start_year) & (df_clean["DATE"].dt.year <= val_end_year)].copy()
    df_test  = df_clean[df_clean["DATE"].dt.year == year].copy()

    X_train = build_features_920(df_train, CHAR_COLS_94, MACRO_COLS, SIC2_DUMMY_COLS)
    X_val   = build_features_920(df_val,   CHAR_COLS_94, MACRO_COLS, SIC2_DUMMY_COLS)
    X_test  = build_features_920(df_test,  CHAR_COLS_94, MACRO_COLS, SIC2_DUMMY_COLS)

    y_train_np = df_train[TARGET].to_numpy(dtype=np.float64)
    y_val_np   = df_val[TARGET].to_numpy(dtype=np.float64)
    y_test_np  = df_test[TARGET].to_numpy(dtype=np.float64)

    X_train_s, x_mu, x_sd = standardize(X_train)
    X_val_s, _, _  = standardize(X_val,  x_mu, x_sd)
    X_test_s, _, _ = standardize(X_test, x_mu, x_sd)

    y_mu = float(y_train_np.mean())
    y_tr_c = (y_train_np - y_mu).astype(np.float32)
    y_va_c = (y_val_np   - y_mu).astype(np.float32)

    X_tr = torch.tensor(X_train_s, dtype=torch.float32, device=DEVICE)
    X_va = torch.tensor(X_val_s,   dtype=torch.float32, device=DEVICE)
    X_te = torch.tensor(X_test_s,  dtype=torch.float32, device=DEVICE)
    y_tr = torch.tensor(y_tr_c,    dtype=torch.float32, device=DEVICE)
    y_va = torch.tensor(y_va_c,    dtype=torch.float32, device=DEVICE)

    del X_train, X_val, X_test, X_train_s, X_val_s, X_test_s
    gc.collect()

    p = X_tr.shape[1]
    XtX = X_tr.T @ X_tr
    Xty = X_tr.T @ y_tr
    n_tr = y_tr.shape[0]

    XtX_64 = XtX.detach().cpu().double().numpy()
    v = np.random.randn(p)
    for _ in range(30):
        v = XtX_64 @ v
        nv = np.linalg.norm(v)
        if nv < 1e-12:
            break
        v /= nv
    max_eig = float(v @ (XtX_64 @ v))
    L = 2.0 * max_eig / n_tr + 1e-12
    gamma_fixed = float(1.0 / L)
    del XtX_64

    try:
        A = XtX.double() / n_tr + 0.01 * torch.eye(p, dtype=torch.float64, device=DEVICE)
        b = Xty.double() / n_tr
        theta_ridge = torch.linalg.solve(A, b).float()
    except Exception:
        theta_ridge = torch.zeros(p, dtype=torch.float32, device=DEVICE)

    val_resid = y_va - X_va @ theta_ridge
    xi = float(val_resid.abs().quantile(XI_QUANTILE).item())
    xi = max(xi, 1e-4)

    if theta_cross_year is not None:
        warm_init = theta_cross_year.clone()
        warm_source = "cross-year"
    else:
        warm_init = torch.zeros(p, dtype=torch.float32, device=DEVICE)
        warm_source = "zeros"

    best_val_loss = float("inf")
    best_lam = LAM_GRID_SORTED[0]
    best_theta_gs = warm_init.clone()
    theta_warm = warm_init.clone()
    gs_iters_total = 0
    gs_converged_cnt = 0

    for lam in LAM_GRID_SORTED:
        theta, info = apg_huber_enet_gpu(
            X_tr, y_tr, lam, RHO, xi,
            gamma=gamma_fixed, max_iter=200, tol=1e-4,
            theta_init=theta_warm, return_info=True
        )
        theta_warm = theta.clone()
        gs_iters_total += info["iters"]
        gs_converged_cnt += int(info["converged"])

        val_loss = huber_loss_t(y_va - X_va @ theta, xi).mean().item()
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_lam = float(lam)
            best_theta_gs = theta.clone()

    best_theta, refit_info = apg_huber_enet_gpu(
        X_tr, y_tr, best_lam, RHO, xi,
        gamma=gamma_fixed, max_iter=5000, tol=1e-5,
        theta_init=best_theta_gs, return_info=True
    )
    theta_cross_year = best_theta.clone()

    y_pred = (X_te @ best_theta).detach().cpu().numpy().astype(np.float64) + y_mu

    res = df_test[["DATE", "permno", "mvel1"]].copy().reset_index(drop=True)
    res["y_true"] = y_test_np
    res["y_pred"] = y_pred
    all_preds.append(res)

    n_nonzero = int((best_theta.abs() > 1e-8).sum().item())
    elapsed = time.time() - t0

    year_info.append({
        "year": year,
        "n_train": len(df_train),
        "n_val": len(df_val),
        "n_test": len(df_test),
        "lam": best_lam,
        "xi": xi,
        "nnz": n_nonzero,
        "sec": elapsed,
        "warm_source": warm_source,
        "gs_avg_iters": gs_iters_total / len(LAM_GRID_SORTED),
        "gs_converged": gs_converged_cnt,
        "refit_iters": refit_info["iters"],
        "refit_converged": refit_info["converged"],
        "refit_restarts": refit_info["n_restarts"],
        "best_val_loss": best_val_loss,
    })

    conv_tag = "✓" if refit_info["converged"] else "x"
    print(
        f"Year {year} | train {len(df_train):>8,} | val {len(df_val):>8,} | test {len(df_test):>7,} | "
        f"lam={best_lam:.2e} xi={xi:.3f} nnz={n_nonzero:>4d}/{p} | "
        f"GS avg {gs_iters_total/len(LAM_GRID_SORTED):.1f}it | refit {refit_info['iters']}it/{refit_info['n_restarts']}rst {conv_tag} | "
        f"{elapsed:.1f}s (warm:{warm_source})"
    )

    del X_tr, X_va, X_te, y_tr, y_va, XtX, Xty, theta_ridge, val_resid, best_theta, best_theta_gs
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\nEstimation complete.")

Year 1987 | train  472,278 | val  764,497 | test  82,404 | lam=3.79e-03 xi=1.899 nnz=  46/920 | GS avg 5.2it | refit 27it/2rst ✓ | 14.3s (warm:zeros)
Year 1988 | train  530,435 | val  788,744 | test  83,415 | lam=3.79e-03 xi=1.324 nnz=  54/920 | GS avg 3.9it | refit 25it/2rst ✓ | 13.6s (warm:cross-year)
Year 1989 | train  588,534 | val  814,060 | test  81,216 | lam=2.64e-03 xi=1.251 nnz=  62/920 | GS avg 4.5it | refit 50it/2rst ✓ | 15.5s (warm:cross-year)
Year 1990 | train  647,363 | val  836,447 | test  80,207 | lam=2.64e-03 xi=1.264 nnz=  64/920 | GS avg 5.1it | refit 49it/2rst ✓ | 16.7s (warm:cross-year)
Year 1991 | train  704,916 | val  859,101 | test  79,274 | lam=3.79e-03 xi=1.320 nnz=  57/920 | GS avg 3.8it | refit 36it/2rst ✓ | 16.5s (warm:cross-year)
Year 1992 | train  761,970 | val  881,321 | test  80,972 | lam=2.64e-03 xi=1.443 nnz=  74/920 | GS avg 4.8it | refit 27it/2rst ✓ | 17.5s (warm:cross-year)
Year 1993 | train  819,497 | val  904,766 | test  86,150 | lam=1.00e-01 xi=

In [7]:
results = pd.concat(all_preds, ignore_index=True)
results["DATE"] = pd.to_datetime(results["DATE"])
results = results.sort_values(["DATE", "permno"]).reset_index(drop=True)

year_info_df = pd.DataFrame(year_info)

print("total predictions:", f"{len(results):,}")
print("test period:", results["DATE"].min().date(), "to", results["DATE"].max().date())

r2_all = oos_r2(results["y_true"], results["y_pred"])

top1000 = (
    results.sort_values(["DATE", "mvel1"], ascending=[True, False])
           .groupby("DATE", sort=False)
           .head(1000)
)
bot1000 = (
    results.sort_values(["DATE", "mvel1"], ascending=[True, True])
           .groupby("DATE", sort=False)
           .head(1000)
)

r2_top = oos_r2(top1000["y_true"], top1000["y_pred"])
r2_bot = oos_r2(bot1000["y_true"], bot1000["y_pred"])

print("=" * 72)
print(f"{'Subsample':<35} {'OOS R^2':>12} {'Paper ENet+H':>18}")
print("-" * 72)
print(f"{'All stocks (panel)':<35} {r2_all*100:>+11.4f}% {'~ +0.11%':>18}")
print(f"{'Top 1,000 (largest mvel1)':<35} {r2_top*100:>+11.4f}% {'~ +0.25%':>18}")
print(f"{'Bottom 1,000 (smallest mvel1)':<35} {r2_bot*100:>+11.4f}% {'~ +0.34%':>18}")
print("=" * 72)

print()
print("lambda median:", f"{year_info_df['lam'].median():.2e}")
print("xi median:", f"{year_info_df['xi'].median():.4f}")
print("nnz median:", int(year_info_df["nnz"].median()))

total predictions: 2,476,033
test period: 1987-01-30 to 2016-12-30
Subsample                                OOS R^2       Paper ENet+H
------------------------------------------------------------------------
All stocks (panel)                      +0.2812%           ~ +0.11%
Top 1,000 (largest mvel1)               +0.1801%           ~ +0.25%
Bottom 1,000 (smallest mvel1)           +0.3190%           ~ +0.34%

lambda median: 1.55e-03
xi median: 1.5099
nnz median: 120


In [8]:
# save a parquet that the backtest notebook can read directly
cols_to_save = ["permno", "DATE", "y_true", "y_pred", "mvel1"]
results[cols_to_save].to_parquet(OUTPUT_PATH, index=False)
year_info_df.to_csv(YEAR_INFO_PATH, index=False)

print("saved prediction parquet:", OUTPUT_PATH)
print("saved yearly diagnostics:", YEAR_INFO_PATH)
results.head()

saved prediction parquet: /content/drive/MyDrive/backtest/enet_huber_results.parquet
saved yearly diagnostics: /content/drive/MyDrive/backtest/enet_huber_year_info.csv


,DATE,permno,mvel1,y_true,y_pred
0,1987-01-30,10000,-0.921324,-0.004300,0.002715
1,1987-01-30,10001,-0.671628,-0.078374,0.005474
2,1987-01-30,10002,-0.413730,-0.018125,0.002891
3,1987-01-30,10003,0.030073,0.007194,-0.002427
4,1987-01-30,10005,-0.994228,0.095700,-0.003038


In [11]:
!pip -q install wrds

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 129.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 95.3 MB/s eta 0:00:00


In [12]:
import os
import pandas as pd
import wrds

CRSP_OUT_PATH = "/content/drive/MyDrive/backtest/crsp_monthly.parquet"

conn = wrds.Connection()

query = """
select
    a.permno,
    a.date,
    a.ret,
    abs(a.prc) * a.shrout as me
from crsp.msf as a
join crsp.msenames as b
    on a.permno = b.permno
   and b.namedt <= a.date
   and a.date <= b.nameendt
where a.date between '1987-01-01' and '2016-12-31'
  and a.ret is not null
  and a.prc is not null
  and a.shrout is not null
  and b.shrcd in (10, 11)
  and b.exchcd in (1, 2, 3)
order by a.date, a.permno
"""

crsp = conn.raw_sql(query, date_cols=["date"])

os.makedirs("/content/drive/MyDrive/backtest", exist_ok=True)
crsp.to_parquet(CRSP_OUT_PATH, index=False)

print("CRSP shape:", crsp.shape)
print("date range:", crsp["date"].min(), "to", crsp["date"].max())
print("Saved to:", CRSP_OUT_PATH)
print(crsp.head())

Enter your WRDS username [root]:zixian_zhou
Enter your password:··········
WRDS recommends setting up a .pgpass file.
Create .pgpass file now [y/n]?: n
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done
CRSP shape: (1917129, 4)
date range: 1987-01-30 00:00:00 to 2016-12-30 00:00:00
Saved to: /content/drive/MyDrive/backtest/crsp_monthly.parquet
   permno       date       ret          me
0   10000 1987-01-30 -0.212121  1581.53125
1   10001 1987-01-30 -0.035714     6689.25
2   10002 1987-01-30   0.09596  15935.9375
3   10003 1987-01-30   0.12987    47523.75
4   10005 1987-01-30  0.666667    722.8125
